In [2]:

from scipy.io import loadmat
import numpy as np
import pandas as pd

# Load your sample session
mat = loadmat("../data/sessionevents01.mat")

t = mat["t"]          # shape (time, 1)
data = mat["data"]    # shape (time, chan, players, events)
labels = mat["labels"]  # shape (events, 8)

print("t shape:", t.shape)
print("data shape:", data.shape)
print("labels shape:", labels.shape)

# Add column names for readability
cols = ["session", "current", "last", "solo",
        "cardValue", "countP1", "countP2", "countP3"]
labels_df = pd.DataFrame(labels, columns=cols)

labels_df.head()


t shape: (501, 1)
data shape: (501, 32, 3, 249)
labels shape: (249, 8)


,session,current,last,solo,cardValue,countP1,countP2,countP3
0,1,3,0,1,0,0,0,0
1,1,1,3,1,0,0,0,1
2,1,2,1,1,0,1,0,1
3,1,2,0,1,0,1,1,1
4,1,3,2,1,0,1,2,1


In [5]:
n_time, n_chan, n_players, n_events = data.shape
print("time:", n_time, "channels:", n_chan,
      "players:", n_players, "events:", n_events)

X_list = []
y_list = []

for e in range(n_events):
    row = labels_df.iloc[e]
    
    for p in range(1, n_players + 1):    # players labeled as 1,2,3
        eeg = data[:, :, p-1, e]        # raw EEG for this player/event
        eeg = np.nan_to_num(eeg)        # replace NaNs
        eeg = eeg.T                     # (chan, time) for CNN
        
        # Assign player role
        if p == row["last"]:
            role = "played"
        elif p == row["current"]:
            role = "next"
        else:
            role = "observer"
        
        X_list.append(eeg)
        y_list.append(role)

# Convert to arrays
X = np.stack(X_list)   # shape (samples, chan, time)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 10 labels:", y[:10])


time: 501 channels: 32 players: 3 events: 249
X shape: (747, 32, 501)
y shape: (747,)
First 10 labels: ['observer' 'observer' 'next' 'next' 'observer' 'played' 'played' 'next'
 'observer' 'observer']


In [6]:
from sklearn.model_selection import train_test_split

# Convert labels to integers for neural nets
label_to_int = {"next": 0, "observer": 1, "played": 2}
y_int = np.array([label_to_int[l] for l in y])

# CNN/EEGNet input shape: (samples, channels, time, 1)
X_dl = X[..., np.newaxis]

print("X_dl shape:", X_dl.shape)
print("y_int[:10]:", y_int[:10])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_dl, y_int, test_size=0.2, random_state=42, stratify=y_int
)

print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes:", X_test.shape, y_test.shape)


X_dl shape: (747, 32, 501, 1)
y_int[:10]: [1 1 0 0 1 2 2 0 1 1]
Train shapes: (597, 32, 501, 1) (597,)
Test shapes: (150, 32, 501, 1) (150,)


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models

def EEGNet(nb_classes, Chans=32, Samples=501, dropoutRate=0.5, F1=8, D=2, F2=16, kernelLength=64):
    """
    EEGNet implementation adapted for your dataset.
    - nb_classes: number of output classes (3: next, observer, played)
    - Chans: number of EEG channels (32)
    - Samples: number of time samples (501)
    """

    input_shape = (Chans, Samples, 1)
    inputs = layers.Input(shape=input_shape)

    # Block 1: Temporal Convolution
    x = layers.Conv2D(
        F1,
        (1, kernelLength),
        padding='same',
        use_bias=False
    )(inputs)
    x = layers.BatchNormalization()(x)

    # Block 2: Depthwise Convolution (spatial filtering across channels)
    x = layers.DepthwiseConv2D(
        (Chans, 1),
        use_bias=False,
        depth_multiplier=D,
        depthwise_constraint=tf.keras.constraints.max_norm(1.)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(dropoutRate)(x)

    # Block 3: Separable Convolution
    x = layers.SeparableConv2D(
        F2,
        (1, 16),
        padding='same',
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropoutRate)(x)

    # Classification block
    x = layers.Flatten()(x)
    outputs = layers.Dense(nb_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model


# Build EEGNet for your dataset
eegnet_model = EEGNet(nb_classes=3, Chans=32, Samples=501)

eegnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

eegnet_model.summary()


Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 32, 501, 1)]      0         
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 501, 8)        512       
                                                                 
 batch_normalization_3 (Batc  (None, 32, 501, 8)       32        
 hNormalization)                                                 
                                                                 
 depthwise_conv2d_1 (Depthwi  (None, 1, 501, 16)       512       
 seConv2D)                                                       
                                                                 
 batch_normalization_4 (Batc  (None, 1, 501, 16)       64        
 hNormalization)                                                 
                                                           

In [8]:
history = eegnet_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)


Epoch 1/50
60/60 [==============================] - 13s 142ms/step - loss: 1.2271 - accuracy: 0.3753 - val_loss: 1.0753 - val_accuracy: 0.4417
Epoch 2/50
60/60 [==============================] - 9s 144ms/step - loss: 1.1201 - accuracy: 0.4130 - val_loss: 1.0595 - val_accuracy: 0.4333
Epoch 3/50
60/60 [==============================] - 8s 127ms/step - loss: 1.0565 - accuracy: 0.4654 - val_loss: 1.0579 - val_accuracy: 0.4583
Epoch 4/50
60/60 [==============================] - 7s 113ms/step - loss: 1.0629 - accuracy: 0.4822 - val_loss: 1.0366 - val_accuracy: 0.4667
Epoch 5/50
60/60 [==============================] - 7s 112ms/step - loss: 1.0227 - accuracy: 0.4675 - val_loss: 1.0200 - val_accuracy: 0.4750
Epoch 6/50
60/60 [==============================] - 4s 71ms/step - loss: 0.9844 - accuracy: 0.5031 - val_loss: 1.0068 - val_accuracy: 0.4750
Epoch 7/50
60/60 [==============================] - 3s 57ms/step - loss: 0.9855 - accuracy: 0.5115 - val_loss: 1.0158 - val_accuracy: 0.4833
Epoch 8

In [9]:
test_loss, test_acc = eegnet_model.evaluate(X_test, y_test, verbose=0)
print("EEGNet Test accuracy:", test_acc)

from sklearn.metrics import classification_report

y_pred = eegnet_model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(
    y_test, y_pred_classes,
    target_names=["next", "observer", "played"]
))


EEGNet Test accuracy: 0.4866666793823242
5/5 [==============================] - 1s 70ms/step
              precision    recall  f1-score   support

        next       0.49      0.52      0.50        50
    observer       0.57      0.52      0.55        67
      played       0.33      0.36      0.35        33

    accuracy                           0.49       150
   macro avg       0.47      0.47      0.47       150
weighted avg       0.49      0.49      0.49       150



In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

def EEGNet_LSTM(nb_classes, Chans=32, Samples=501,
                dropoutRate=0.5, F1=8, D=2, F2=16, kernelLength=64,
                lstm_units=32):
    """
    EEGNet + LSTM hybrid:
    - First part: EEGNet conv blocks (temporal + spatial + separable conv)
    - Then: reshape feature maps into a time sequence and feed to LSTM
    """

    input_shape = (Chans, Samples, 1)
    inputs = layers.Input(shape=input_shape)

    # ----- EEGNet blocks (same as before, up to last Dropout) -----
    # Block 1: Temporal Convolution
    x = layers.Conv2D(
        F1,
        (1, kernelLength),
        padding='same',
        use_bias=False
    )(inputs)
    x = layers.BatchNormalization()(x)

    # Block 2: Depthwise Convolution (spatial)
    x = layers.DepthwiseConv2D(
        (Chans, 1),
        use_bias=False,
        depth_multiplier=D,
        depthwise_constraint=tf.keras.constraints.max_norm(1.)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(dropoutRate)(x)

    # Block 3: Separable Convolution
    x = layers.SeparableConv2D(
        F2,
        (1, 16),
        padding='same',
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropoutRate)(x)

    # ----- LSTM part -----
    # Shape after last pooling is (batch, 1, Samples/32, F2)
    # We reshape to (batch, time_steps, features)
    time_steps = Samples // 32  # 501 // 32 = 15
    x = layers.Reshape((time_steps, F2))(x)   # (batch, 15, F2)

    x = layers.LSTM(lstm_units, return_sequences=False)(x)
    x = layers.Dropout(dropoutRate)(x)

    outputs = layers.Dense(nb_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model


# Build the EEGNet-LSTM model for your data
eegnet_lstm_model = EEGNet_LSTM(nb_classes=3, Chans=32, Samples=501)

eegnet_lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

eegnet_lstm_model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 501, 1)]      0         
                                                                 
 conv2d (Conv2D)             (None, 32, 501, 8)        512       
                                                                 
 batch_normalization (BatchN  (None, 32, 501, 8)       32        
 ormalization)                                                   
                                                                 
 depthwise_conv2d (Depthwise  (None, 1, 501, 16)       512       
 Conv2D)                                                         
                                                                 
 batch_normalization_1 (Batc  (None, 1, 501, 16)       64        
 hNormalization)                                                 
                                                             

In [1]:
print("Has eegnet_lstm_model?", "eegnet_lstm_model" in globals())
print("Has model?", "model" in globals())


Has eegnet_lstm_model? False
Has model? False


In [2]:
from tensorflow.keras.callbacks import EarlyStopping

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_lstm = eegnet_lstm_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early],
    verbose=1
)


NameError: name 'eegnet_lstm_model' is not defined

In [3]:
from tensorflow.keras.callbacks import EarlyStopping

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_lstm = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early],
    verbose=1
)


NameError: name 'model' is not defined

In [4]:
from sklearn.metrics import classification_report

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("EEGNet-LSTM Test accuracy:", test_acc)

y_pred = model.predict(X_test, batch_size=8)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(
    y_test, y_pred_classes,
    target_names=["next", "observer", "played"]
))


NameError: name 'model' is not defined

In [5]:
eegnet_lstm_model = EEGNet_LSTM(nb_classes=3, Chans=32, Samples=501)

eegnet_lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

eegnet_lstm_model.summary()


NameError: name 'EEGNet_LSTM' is not defined

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

def EEGNet_LSTM(nb_classes, Chans=32, Samples=501,
                dropoutRate=0.5, F1=8, D=2, F2=16, kernelLength=64,
                lstm_units=32):

    input_shape = (Chans, Samples, 1)
    inputs = layers.Input(shape=input_shape)

    # Block 1: Temporal Convolution
    x = layers.Conv2D(F1, (1, kernelLength), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)

    # Block 2: Depthwise (spatial) Convolution
    x = layers.DepthwiseConv2D(
        (Chans, 1),
        use_bias=False,
        depth_multiplier=D,
        depthwise_constraint=tf.keras.constraints.max_norm(1.)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(dropoutRate)(x)

    # Block 3: Separable Convolution
    x = layers.SeparableConv2D(F2, (1, 16), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropoutRate)(x)

    # LSTM expects (batch, time_steps, features)
    time_steps = Samples // 32  # because pooling is 4 then 8 => /32 overall
    x = layers.Reshape((time_steps, F2))(x)

    x = layers.LSTM(lstm_units, return_sequences=False)(x)
    x = layers.Dropout(dropoutRate)(x)

    outputs = layers.Dense(nb_classes, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model


In [7]:
model = EEGNet_LSTM(nb_classes=3, Chans=32, Samples=501)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 32, 501, 1)]      0         
                                                                 
 conv2d (Conv2D)             (None, 32, 501, 8)        512       
                                                                 
 batch_normalization (BatchN  (None, 32, 501, 8)       32        
 ormalization)                                                   
                                                                 
 depthwise_conv2d (Depthwise  (None, 1, 501, 16)       512       
 Conv2D)                                                         
                                                                 
 batch_normalization_1 (Batc  (None, 1, 501, 16)       64        
 hNormalization)                                                 
                                                             

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_lstm = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early],
    verbose=1
)


NameError: name 'X_train' is not defined

In [9]:
from scipy.io import loadmat
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data
mat = loadmat("../data/sessionevents01.mat")
data = mat["data"]
labels = mat["labels"]

cols = ["session", "current", "last", "solo",
        "cardValue", "countP1", "countP2", "countP3"]
labels_df = pd.DataFrame(labels, columns=cols)

# Build X, y
n_time, n_chan, n_players, n_events = data.shape

X_list, y_list = [], []
for e in range(n_events):
    row = labels_df.iloc[e]
    for p in range(1, n_players + 1):
        eeg = data[:, :, p-1, e]     # (time, chan)
        eeg = np.nan_to_num(eeg)
        eeg = eeg.T                  # (chan, time)

        if p == row["last"]:
            role = "played"
        elif p == row["current"]:
            role = "next"
        else:
            role = "observer"

        X_list.append(eeg)
        y_list.append(role)

X = np.stack(X_list)                 # (samples, chan, time)
y = np.array(y_list)

# Convert labels to ints + add channel dim
label_to_int = {"next": 0, "observer": 1, "played": 2}
y_int = np.array([label_to_int[l] for l in y])
X_dl = X[..., np.newaxis]            # (samples, chan, time, 1)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_dl, y_int, test_size=0.2, random_state=42, stratify=y_int
)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


X_train: (597, 32, 501, 1) y_train: (597,)
X_test: (150, 32, 501, 1) y_test: (150,)


In [10]:
from tensorflow.keras.callbacks import EarlyStopping

early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_lstm = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early],
    verbose=1
)


Epoch 1/50
60/60 [==============================] - 22s 172ms/step - loss: 1.0692 - accuracy: 0.4256 - val_loss: 1.0945 - val_accuracy: 0.3667
Epoch 2/50
60/60 [==============================] - 5s 79ms/step - loss: 1.0774 - accuracy: 0.4109 - val_loss: 1.0761 - val_accuracy: 0.4333
Epoch 3/50
60/60 [==============================] - 5s 78ms/step - loss: 1.0601 - accuracy: 0.4507 - val_loss: 1.0691 - val_accuracy: 0.4417
Epoch 4/50
60/60 [==============================] - 4s 69ms/step - loss: 1.0270 - accuracy: 0.4927 - val_loss: 1.0489 - val_accuracy: 0.4417
Epoch 5/50
60/60 [==============================] - 4s 65ms/step - loss: 1.0384 - accuracy: 0.4591 - val_loss: 1.0207 - val_accuracy: 0.4417
Epoch 6/50
60/60 [==============================] - 4s 63ms/step - loss: 1.0093 - accuracy: 0.4738 - val_loss: 1.0206 - val_accuracy: 0.4500
Epoch 7/50
60/60 [==============================] - 4s 66ms/step - loss: 1.0123 - accuracy: 0.5115 - val_loss: 1.0188 - val_accuracy: 0.5000
Epoch 8/50


In [11]:
print("X_train in memory?", "X_train" in globals())


X_train in memory? True


In [ ]:
from sklearn.metrics import classification_report

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("EEGNet-LSTM Test accuracy:", test_acc)

y_pred = model.predict(X_test, batch_size=8)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(y_test, y_pred_classes,
                           target_names=["next","observer","played"]))


EEGNet-LSTM Test accuracy: 0.5066666603088379
19/19 [==============================] - 3s 38ms/step
              precision    recall  f1-score   support

        next       0.47      0.44      0.45        50
    observer       0.58      0.67      0.62        67
      played       0.35      0.27      0.31        33

    accuracy                           0.51       150
   macro avg       0.47      0.46      0.46       150
weighted avg       0.49      0.51      0.50       150



: 